# DepthWizard: fine-tune DA3-Mono-Large on GAMUS (Kaggle)

**You run this notebook.** Kaggle settings: **Accelerator = GPU (T4 x2 or P100)**, **Internet = On**.

Before running:
1. On your laptop, run `bash scripts/make_kaggle_bundle.sh`. This creates `dist/depthwizard-code.zip`.
2. Upload that zip on Kaggle as a **Dataset** named `depthwizard-code`, then add it to this notebook with *Add Input*.

What this does: installs the pinned DA3 commit, downloads a GAMUS subset (DC + PHL train, NYC held out), trains with
`configs/finetune_da3_kaggle.yaml`, and saves `best.pt` + `metrics.json` (with the checkpoint sha256) to the output.
GAMUS is CC-BY-4.0 and needs no account; DA3MONO-LARGE is Apache-2.0.

**Untested parts (THIS SHOULD BE TESTED on the first run):** the DA3 gradient-checkpointing wrapper; how DA3's `numpy<2`
pin interacts with Kaggle's preinstalled packages. The notebook stops if installing DA3 changes torch.

In [ ]:
!nvidia-smi
import torch, sys; print(torch.__version__, torch.version.cuda, torch.cuda.is_available(), sys.version)

In [ ]:
import os, shutil, zipfile, glob
src = glob.glob('/kaggle/input/depthwizard-code/**/depthwizard-code.zip', recursive=True) or glob.glob('/kaggle/input/depthwizard-code*')
print(src)
os.makedirs('/kaggle/working/DepthWizard', exist_ok=True)
if src and src[0].endswith('.zip'):
    zipfile.ZipFile(src[0]).extractall('/kaggle/working/DepthWizard')
else:  # Kaggle may have unzipped the dataset already
    shutil.copytree(src[0], '/kaggle/working/DepthWizard', dirs_exist_ok=True)
%cd /kaggle/working/DepthWizard
!ls

In [ ]:
import torch; TORCH_BEFORE = torch.__version__
# requirements.txt has no torch line: Kaggle's CUDA torch stays in place
!pip install -q -r setup/requirements.txt
!git clone -q https://github.com/ByteDance-Seed/depth-anything-3 third_party/depth-anything-3
!git -C third_party/depth-anything-3 checkout -q 3d835ec1a5802d64a8b8b15f817a1ab54809bfe4
!pip install -q -e third_party/depth-anything-3
import subprocess, sys
after = subprocess.run([sys.executable, '-c', 'import torch;print(torch.__version__)'], capture_output=True, text=True).stdout.strip()
print('torch before', TORCH_BEFORE, 'after', after)
assert after == TORCH_BEFORE, 'DA3 install changed torch - stop and fix (reinstall Kaggle torch or pin), do not force'

In [ ]:
# ~9.3 MB per tile. per-city 250 -> 500 train tiles ~4.7 GB; NYC 200 test tiles ~1.9 GB (Kaggle /kaggle/working = 20 GB)
!PYTHONPATH=src python scripts/fetch_gamus_subset.py --split train --per-city 250 --cities DC,PHL --out /kaggle/working/gamus --max-gb 6
!PYTHONPATH=src python scripts/fetch_gamus_subset.py --split test --per-city 200 --cities NYC --out /kaggle/working/gamus --max-gb 3
!du -sh /kaggle/working/gamus

In [ ]:
!PYTHONPATH=src python -m unittest discover -s tests 2>&1 | tail -3 || true
!PYTHONPATH=src python training/finetune.py --config configs/finetune_da3_kaggle.yaml

In [ ]:
import json, glob
m = sorted(glob.glob('/kaggle/working/runs/*/metrics.json'))[-1]
r = json.load(open(m))
print(m)
print({k: r[k] for k in ('zero_shot_val_median_fitted_rmse_m', 'best_val_median_fitted_rmse_m', 'loss_first_20pct_mean', 'loss_last_20pct_mean', 'checkpoints')})
# Download best.pt + metrics.json from the notebook Output tab. Verify the sha256 locally before use.